# Phase 10: Testing and Professional Skills
## Proving your code works

In this final phase, you learn to write tests for mini-odibi using **pytest**. 
You also learn debugging, CLI basics, and code organization.

Testing is what separates hobby code from professional code. 
In interviews, being able to talk about testing shows maturity.

---
## Section 1: What is pytest?

**pytest** is Python's most popular testing framework. It is what Odibi uses.

A test is a function that:
1. Sets up some data (Arrange)
2. Calls the code you want to test (Act)
3. Checks the result is correct (Assert)

This is called the **Arrange-Act-Assert** pattern.

### Installing pytest
```bash
pip install pytest
```

### Running tests
```bash
pytest tests/ -v            # Run all tests, verbose
pytest tests/test_config.py  # Run one file
pytest tests/ -k "test_name" # Run tests matching a pattern
```

---
## Section 2: Writing Your First Test


In [ ]:
# Save this as learning/tests/test_example.py
# Then run: pytest learning/tests/test_example.py -v

def test_addition():
    """Test that basic addition works."""
    result = 2 + 2
    assert result == 4

def test_string_upper():
    """Test string uppercase."""
    assert "hello".upper() == "HELLO"

def test_list_length():
    """Test list length."""
    items = [1, 2, 3]
    assert len(items) == 3

# assert is the KEY. If the expression after assert is True, the test passes.
# If it is False, the test FAILS and pytest shows you what went wrong.

---
## Section 3: Testing with pytest.raises

Sometimes you want to test that code RAISES an error.

In [ ]:
import pytest

def divide(a, b):
    if b == 0:
        raise ValueError("Cannot divide by zero")
    return a / b

def test_divide_normal():
    assert divide(10, 2) == 5.0

def test_divide_by_zero():
    """Test that dividing by zero raises ValueError."""\n
    with pytest.raises(ValueError, match="Cannot divide by zero"):
        divide(10, 0)

---
## Section 4: Fixtures

A **fixture** is a function that provides test data or setup. 
Instead of repeating setup code in every test, you define it once.

In [ ]:
import pytest
import pandas as pd

@pytest.fixture
def sample_df():
    """Create a sample DataFrame for testing."""\n
    return pd.DataFrame([
        {"id": 1, "name": "Alice", "dept": "Eng"},
        {"id": 2, "name": "Bob", "dept": "Mkt"},
        {"id": 3, "name": "Charlie", "dept": "Eng"},
    ])

def test_row_count(sample_df):
    """Test that we have the right number of rows."""\n
    assert len(sample_df) == 3

def test_columns(sample_df):
    """Test that expected columns exist."""\n
    assert "id" in sample_df.columns
    assert "name" in sample_df.columns

def test_filter(sample_df):
    """Test filtering by department."""\n
    eng = sample_df[sample_df["dept"] == "Eng"]
    assert len(eng) == 2

---
## Section 5: Parametrize

Run the same test with multiple inputs.

In [ ]:
import pytest

@pytest.mark.parametrize("mode,valid", [
    ("overwrite", True),
    ("append", True),
    ("upsert", True),
    ("delete", False),
    ("", False),
])
def test_write_mode_validation(mode, valid):
    """Test write mode validation with multiple inputs."""\n
    valid_modes = ["overwrite", "append", "upsert", "append_once", "merge"]
    result = mode in valid_modes
    assert result == valid

---
## Section 6: Write Tests for Mini-Odibi

Now write real tests for the code you built in Phases 8-9.

Create these test files in `learning/tests/`:

In [ ]:
# learning/tests/test_config.py
# YOUR IMPLEMENTATION:
#
# Test that:
# - Valid config creates successfully
# - Empty name raises ValidationError
# - Invalid engine type raises error
# - Upsert without keys raises error
# - Config loads from a YAML string


In [ ]:
# learning/tests/test_engine.py
# YOUR IMPLEMENTATION:
#
# Test that:
# - PandasEngine can read a CSV file
# - PandasEngine returns correct row count
# - PandasEngine returns correct column list
# - Reading a non-existent file raises an error
# - Use a fixture that creates a temp CSV file


In [ ]:
# learning/tests/test_transformers.py
# YOUR IMPLEMENTATION:
#
# Test that:
# - rename_columns renames correctly
# - filter_rows filters correctly
# - drop_columns drops correctly
# - TransformRegistry can register and retrieve functions


---
## Section 7: Mocking with @patch and MagicMock

This is the **#1 pattern in odibi's test suite** (used in 70+ test files).

When you test a function that calls external systems (file I/O, databases, APIs),
you don't want to actually call them. You **mock** them — replace the real thing
with a fake that you control.

### MagicMock: The Universal Fake Object

A `MagicMock` can pretend to be anything. It accepts any attribute access
and any method call without errors.

In [ ]:
from unittest.mock import MagicMock

# MagicMock pretends to be anything
fake_connection = MagicMock()

# You can call any method on it - it won't error
fake_connection.connect()
fake_connection.read_file("data.csv")

# You can set what it returns
fake_connection.get_path.return_value = "/fake/path/data"
result = fake_connection.get_path("sales.csv")
print(f"Mocked path: {result}")  # /fake/path/data

# You can check if it was called
fake_connection.get_path.assert_called_once_with("sales.csv")
print("✅ get_path was called with 'sales.csv'")

# You can check how many times it was called
print(f"connect() called {fake_connection.connect.call_count} time(s)")

### @patch: Replace a Real Function During a Test

`@patch` temporarily swaps out a real function/class with a MagicMock.
The key rule: **patch where it's USED, not where it's DEFINED.**

This is the #1 mistake in odibi tests. If `node.py` does `from odibi.connections import LocalConnection`,
you patch `odibi.node.LocalConnection`, NOT `odibi.connections.LocalConnection`.

In [ ]:
from unittest.mock import patch, MagicMock

# --- The code we want to test ---
class DataReader:
    def __init__(self, connection):
        self.connection = connection

    def read(self, path):
        """Read data using the connection."""
        full_path = self.connection.get_path(path)
        return self.connection.read_file(full_path)

# --- The test ---
def test_data_reader():
    # Create a fake connection
    mock_conn = MagicMock()
    mock_conn.get_path.return_value = "/data/sales.csv"
    mock_conn.read_file.return_value = [{"id": 1, "name": "Alice"}]

    # Use the fake connection
    reader = DataReader(mock_conn)
    result = reader.read("sales.csv")

    # Verify behavior
    assert result == [{"id": 1, "name": "Alice"}]
    mock_conn.get_path.assert_called_once_with("sales.csv")
    mock_conn.read_file.assert_called_once_with("/data/sales.csv")
    print("✅ DataReader test passed")

test_data_reader()

# --- Using @patch decorator (how odibi tests do it) ---
# @patch replaces a module-level name during the test.
# The mock is passed as an extra argument to your test function.

import os

def get_file_size(path):
    """Get file size - calls os.path.getsize internally."""
    return os.path.getsize(path)

# We patch os.path.getsize so we don't need a real file
@patch("os.path.getsize", return_value=1024)
def test_get_file_size(mock_getsize):
    result = get_file_size("fake_file.csv")
    assert result == 1024
    mock_getsize.assert_called_once_with("fake_file.csv")
    print("✅ @patch test passed")

test_get_file_size()

---
## Section 8: monkeypatch (Setting Environment Variables)

pytest's built-in `monkeypatch` fixture lets you temporarily set/unset
environment variables. Odibi uses this for testing `${VAR}` substitution
in YAML configs.

In [ ]:
import os

# In a real test file, monkeypatch is a pytest fixture.
# Here we'll show the pattern you'd use:

# def test_env_substitution(monkeypatch):
#     monkeypatch.setenv("DB_HOST", "localhost")
#     monkeypatch.setenv("DB_PORT", "5432")
#
#     config = load_yaml_with_env("config.yaml")
#     assert config["host"] == "localhost"
#
# def test_missing_env_raises(monkeypatch):
#     monkeypatch.delenv("DB_HOST", raising=False)  # Ensure it's not set
#
#     with pytest.raises(ValueError, match="Missing environment variable"):
#         load_yaml_with_env("config.yaml")

# monkeypatch is SAFER than os.environ because it automatically
# restores the original value after the test ends.

# Real odibi example from test_config_loader.py:
# def test_load_yaml_with_env_success(monkeypatch):
#     monkeypatch.setenv("TEST_VAR", "value1")
#     config = load_yaml_with_env(path)
#     assert config["key"] == "value1"

print("monkeypatch: setenv(), delenv(), setattr(), delattr()")
print("All changes are automatically undone after the test")

---
## Section 9: tempfile and tmp_path (File-Based Tests)

When testing code that reads/writes files, create temp files that
get cleaned up automatically.

In [ ]:
import tempfile
import os

# Method 1: tempfile.NamedTemporaryFile (used throughout odibi tests)
with tempfile.NamedTemporaryFile(mode="w", delete=False, suffix=".yaml") as f:
    f.write("key: value\nname: test")
    path = f.name

print(f"Temp file created at: {path}")
print(f"File exists: {os.path.exists(path)}")

# Read it back
with open(path) as f:
    print(f"Contents: {f.read()}")

# Clean up (in real tests, use try/finally)
os.remove(path)

# Method 2: tempfile.TemporaryDirectory (for multi-file tests)
with tempfile.TemporaryDirectory() as tmpdir:
    # Create files in the temp directory
    config_path = os.path.join(tmpdir, "config.yaml")
    env_path = os.path.join(tmpdir, "env.dev.yaml")

    with open(config_path, "w") as f:
        f.write("project: test")
    with open(env_path, "w") as f:
        f.write("engine: pandas")

    print(f"Created {os.listdir(tmpdir)}")
    # Both files auto-deleted when 'with' block ends

# Method 3: pytest's tmp_path fixture (cleanest)
# def test_read_config(tmp_path):
#     config_file = tmp_path / "config.yaml"
#     config_file.write_text("project: test")
#     config = load_yaml_with_env(str(config_file))
#     assert config["project"] == "test"

print("\n✅ tmp_path is the cleanest -- pytest handles all cleanup")

---
## Section 10: Test Classes (Grouping Related Tests)

Odibi groups related tests into classes. This keeps things organized
and lets you share setup logic.

In [ ]:
import pytest

# Group related tests into a class (no __init__ needed)
class TestEnvironmentsOverride:
    """Tests for environment override functionality."""

    def test_override_connections(self):
        """Test that connections can be overridden."""
        base = {"type": "local", "path": "./dev"}
        override = {"path": "/prod/data"}
        merged = {**base, **override}
        assert merged["path"] == "/prod/data"
        assert merged["type"] == "local"  # preserved

    def test_override_engine(self):
        """Test that engine can be overridden."""
        base_engine = "pandas"
        prod_engine = "spark"
        assert prod_engine != base_engine

    def test_no_override_uses_base(self):
        """Test that missing override keeps base value."""
        base = {"engine": "pandas", "retry": False}
        override = {}  # empty override
        merged = {**base, **override}
        assert merged["engine"] == "pandas"

# Run with: pytest -v
# Output:
# TestEnvironmentsOverride::test_override_connections PASSED
# TestEnvironmentsOverride::test_override_engine PASSED
# TestEnvironmentsOverride::test_no_override_uses_base PASSED

print("✅ Test classes group related tests and show nicely in pytest output")

---
## Section 11: Real Odibi Test Patterns

Here are the exact patterns you'll see in odibi's test suite.
Study these — they cover 90% of what you need.

In [ ]:
# Pattern 1: Testing a config model with Pydantic
# from odibi.config import ProjectConfig
#
# def test_valid_config():
#     config = ProjectConfig(**valid_config_dict)
#     assert config.project == "test"
#
# def test_invalid_config_raises():
#     with pytest.raises(ValidationError):
#         ProjectConfig(**bad_config_dict)

# Pattern 2: Mocking a connection in node tests
# @patch("odibi.node.LocalConnection")
# def test_node_read(MockConn):
#     mock_conn = MockConn.return_value
#     mock_conn.read.return_value = sample_df
#     node = Node(config, connections={"store": mock_conn})
#     result = node.execute()
#     assert len(result) == 3

# Pattern 3: Testing with temp YAML files
# def test_env_override():
#     with tempfile.TemporaryDirectory() as tmpdir:
#         main_path = os.path.join(tmpdir, "config.yaml")
#         env_path = os.path.join(tmpdir, "env.dev.yaml")
#         with open(main_path, "w") as f:
#             f.write(yaml_content)
#         with open(env_path, "w") as f:
#             f.write(env_content)
#         config = load_yaml_with_env(main_path, env="dev")
#         assert config["engine"] == "pandas"

# Pattern 4: Testing a transformer
# def test_rename_columns():
#     df = pd.DataFrame({"old_name": [1, 2, 3]})
#     result = rename_columns(df, mapping={"old_name": "new_name"})
#     assert "new_name" in result.columns
#     assert "old_name" not in result.columns

print("These 4 patterns cover most odibi tests.")
print("Pattern 1: Config validation (pytest.raises + Pydantic)")
print("Pattern 2: Mocking connections (@patch + MagicMock)")
print("Pattern 3: File-based tests (tempfile + load_yaml_with_env)")
print("Pattern 4: Transformer tests (create df → transform → assert)")

---
## Section 12: Debugging

When a test fails, you need to figure out why. Python gives you tools:

In [ ]:
# breakpoint() -- drops you into the Python debugger
# Insert this line where you want to pause:
#   breakpoint()
#
# Then run your code normally. It will pause at that line.
# In the debugger:
#   n = next line
#   c = continue (run until next breakpoint)
#   p variable = print a variable's value
#   q = quit debugger

# Useful pytest flags for debugging:
# pytest -v                    # Verbose output
# pytest --tb=short            # Shorter tracebacks
# pytest --tb=long             # Full tracebacks
# pytest -x                    # Stop on first failure
# pytest -k "test_name"        # Run only matching tests
# pytest --pdb                 # Drop into debugger on failure
# pytest -s                    # Show print() output (not captured)

# Example:
def buggy_function(items):
    total = 0
    for item in items:
        # breakpoint()  # Uncomment to debug
        total += item
    return total

result = buggy_function([1, 2, 3])
print(f"Result: {result}")

---
## Final Checkpoint -- You Are Done

You have completed the entire program. Here is what you built:

1. **Python fundamentals** -- variables, types, functions, error handling
2. **Data structures** -- lists, dicts, sets, comprehensions
3. **Standard library** -- os, pathlib, json, re, logging, datetime, collections, enum
4. **OOP** -- classes, inheritance, ABC, dunder methods
5. **Advanced patterns** -- decorators, generators, context managers
6. **Pydantic** -- type-safe configuration with validation
7. **Pandas** -- DataFrame operations, groupby, merge
8. **Mini-odibi core** -- config, engine, connections, nodes
9. **Mini-odibi features** -- registry, transformers, validation, pipeline
10. **Testing** -- pytest, fixtures, parametrize, mocking, monkeypatch, tempfile, debugging

You wrote every line yourself. You understand every design pattern because you implemented it.

### What to do next

1. **Add the Polars engine** -- implement `PolarsEngine(BaseEngine)` to reinforce the ABC pattern
2. **Read the real Odibi code** -- you will now understand it. Start with `odibi/node.py`
3. **Practice interview problems** -- use the drills in each notebook, add LeetCode SQL
4. **Build a real pipeline** -- use the real Odibi to build something for your team

You are ready.